# Sprint 3 — Serving & Integration
### Wrap the model behind a FastAPI server with batch inference

The service itself lives in `../serving/` (a server is a Python package, not a notebook).
This notebook works through the five items on the Sprint 3 checklist using the real app.
It runs **in-process** with FastAPI's `TestClient`, so you don't need to start a server.

| # | Checklist item | Section |
|---|---|---|
| 1 | FastAPI REST service architecture for ML | §1 |
| 2 | Request/response payload schemas (Pydantic) | §2 |
| 3 | Low-latency inference & model caching | §3 |
| 4 | Error handling & batch prediction endpoints | §4 |
| 5 | Connecting the ML endpoint to Frontend/Backend squad apps | §5 |

**Prerequisites:** run Sprint 1's `preprocess.ipynb` and `embeddings.ipynb` first, then
`pip install -r ../serving/requirements.txt`.

In [ ]:
%run config.ipynb

In [ ]:
import os, sys, time
import pandas as pd

SERVING_DIR = (PROJECT_ROOT / ".." / "serving").resolve()
sys.path.insert(0, str(SERVING_DIR))
sys.path.insert(0, str(SERVING_DIR / "clients"))

# Point the service at this pipeline's artifacts. Set SCREENING_MODEL_NAME=hashing
# to try the API offline with a stand-in encoder (its scores are not meaningful).
os.environ.setdefault("SCREENING_ARTIFACTS_DIR", str(PROCESSED_DATA_DIR))

from fastapi.testclient import TestClient
from app.main import create_app
from app.settings import get_settings

import logging
logging.getLogger("screening.api").setLevel(logging.WARNING)  # hide per-request access logs here

settings = get_settings()
print("Model:", settings.model_name)
print("Artifacts:", settings.artifacts_dir)

## 1. FastAPI REST service architecture for ML

```
client ──JSON──▶ middleware (request-id, timing, CORS, gzip)
               ▶ route         Pydantic validates payload        → 422 if malformed
               ▶ MatchingService  clean → cache → batch encode → cosine → top-k
               ▶ response      Pydantic serialises; errors share one envelope
```

Key decisions (see `serving/app/main.py`):
* **Lifespan startup.** The model and embeddings are loaded once, before any request is accepted.
* **Plain `def` endpoints.** Inference is CPU-bound, so FastAPI runs these endpoints in its
  threadpool and the event loop stays responsive.
* **Versioned `/v1` routes** plus `/health` and `/ready` probes.

Entering the `with` block below runs the lifespan, which loads the model.

In [ ]:
app = create_app(settings)
client = TestClient(app, raise_server_exceptions=False)
t0 = time.perf_counter()
client.__enter__()          # runs startup: model + embeddings loaded once
print(f"Startup took {time.perf_counter() - t0:.1f}s")

pd.DataFrame(
    [(", ".join(sorted(r.methods - {"HEAD"})), r.path, r.summary or "")
     for r in app.routes if hasattr(r, "methods") and r.path.startswith(("/v1", "/health", "/ready"))],
    columns=["method", "path", "summary"],
)

In [ ]:
info = client.get("/v1/model").json()
info

`job_embeddings_source` / `cohort_embeddings_source` should read `loaded ...npy`.
That means the service reused Sprint 1's saved embeddings: a one-row spot check confirmed
they came from the same model, so nothing had to be re-encoded.

## 2. Request/response payload schemas (Pydantic)
`serving/app/schemas.py` defines the contract. FastAPI uses these models to validate input
and to generate the OpenAPI spec that the other squads build against.

In [ ]:
from app.schemas import MatchRequest, ScreenRequest
import json
print(json.dumps(MatchRequest.model_json_schema(), indent=2))

In [ ]:
# The validation rules at work. Each of these fails before any model code runs:
bad_payloads = {
    "missing text":       {},
    "wrong type":         {"text": 123},
    "top_k out of range": {"text": "Accountant with ten years of payroll and tax experience", "top_k": 99},
    "typo'd field name":  {"txt": "Accountant with ten years of payroll and tax experience"},
}
rows = []
for name, payload in bad_payloads.items():
    r = client.post("/v1/match", json=payload)
    details = r.json()["error"]["details"]
    rows.append({"case": name, "status": r.status_code,
                 "fields": ", ".join(d["field"] for d in details),
                 "messages": " | ".join(d["message"] for d in details)})
pd.DataFrame(rows)

In [ ]:
# A cross-field rule: a job needs exactly ONE of category / description.
r = client.post("/v1/screen", json={"job": {}, "candidates": [{"text": "x" * 50}]})
r.status_code, r.json()["error"]["details"]

## 3. Low-latency inference & model caching
Three things to measure:
1. **Cold vs. warm.** The second request for the same resume hits the LRU embedding cache and skips the model.
2. **One-by-one vs. batch.** 32 resumes in one `/v1/match/batch` call need one forward pass, not 32.
3. **Cohort ranking.** Searching all 2,481 resumes is a single matrix product on pre-loaded embeddings.

In [ ]:
resumes_clean = pd.read_csv(PROCESSED_RESUMES_CSV)
sample = resumes_clean.sample(32, random_state=RANDOM_SEED).reset_index(drop=True)
tag = f" [run {time.time_ns()}]"   # unique text so the first pass really is a cache miss

def ms(fn):
    t0 = time.perf_counter(); out = fn(); return (time.perf_counter() - t0) * 1000, out

cold = [ms(lambda t=t: client.post("/v1/match", json={"text": t + tag}))[0] for t in sample.clean_text]
warm = [ms(lambda t=t: client.post("/v1/match", json={"text": t + tag}))[0] for t in sample.clean_text]

items = [{"id": str(i), "text": t + tag + " batch"} for i, t in enumerate(sample.clean_text)]
one_by_one = sum(ms(lambda it=it: client.post("/v1/match", json={"text": it["text"] + " solo"}))[0] for it in items)
batch_ms, _ = ms(lambda: client.post("/v1/match/batch", json={"items": items}))
rank_ms, _ = ms(lambda: client.post("/v1/cohort/rank", json={"job": {"category": "ENGINEERING"}, "top_k": 10}))

latency = pd.DataFrame([
    {"scenario": "single /v1/match, cold (cache miss)", "ms": pd.Series(cold).median(), "stat": "p50"},
    {"scenario": "single /v1/match, warm (cache hit)",  "ms": pd.Series(warm).median(), "stat": "p50"},
    {"scenario": f"{len(items)} resumes, one request each", "ms": one_by_one, "stat": "total"},
    {"scenario": f"{len(items)} resumes, one batch request", "ms": batch_ms, "stat": "total"},
    {"scenario": "rank 2,481-resume cohort", "ms": rank_ms, "stat": "single"},
]).round(1)
print(f"Batch speed-up: {one_by_one / batch_ms:.1f}x   Cache speed-up: {pd.Series(cold).median() / pd.Series(warm).median():.1f}x")
latency

In [ ]:
client.get("/v1/model").json()["cache"]

## 4. Error handling & batch prediction endpoints
The same messy inputs from Sprint 2's `edge_cases.ipynb`, sent through the API.

* As a **batch**, the request still succeeds. Each bad item comes back as
  `rejected` with a reason, and good items are scored as usual.
* As **single** requests, each bad item gets a 422 with a stable error `code`.

In [ ]:
edge_cases = {
    "empty_string": "",
    "whitespace_only": "    \n\t  ",
    "very_short": "Nurse.",
    "non_english_snippet": "Ingeniero de software con experiencia en Python y machine learning",
    "special_characters_only": "!!!@@@###$$$%%%^^^&&&***",
    "html_resume": "<div><h1>Accountant</h1><p>Payroll, tax returns &amp; general ledger reconciliation.</p></div>",
    "out_of_scope_musician": "Professional saxophonist with 10 years experience performing jazz concerts, "
                             "touring internationally, and teaching music theory at a conservatory.",
    "normal_resume": resumes_clean.iloc[0]["clean_text"],
}
batch = client.post("/v1/match/batch", json={"items": [{"id": k, "text": v} for k, v in edge_cases.items()]})
print("HTTP", batch.status_code, "| summary:", batch.json()["summary"])
pd.DataFrame([
    {"id": r["id"], "status": r["status"], "best_category": r["best_category"],
     "similarity": r["similarity"], "warning": (r["warnings"] or [""])[0][:90]}
    for r in batch.json()["results"]
])

In [ ]:
# The same inputs, one at a time. The single endpoint answers with HTTP errors instead.
pd.DataFrame([
    {"input": k, "http": (r := client.post("/v1/match", json={"text": v})).status_code,
     "code": r.json().get("error", {}).get("code", "—")}
    for k, v in edge_cases.items()
])

In [ ]:
# Problems with the request as a whole (not with one item) fail the entire batch:
too_big = client.post("/v1/match/batch", json={"items": [{"text": "x" * 40}] * (settings.max_batch_size + 1)})
typo_category = client.post("/v1/screen", json={"job": {"category": "ACOUNTANT"},
                                                "candidates": [{"text": "x" * 40}]})
too_big.status_code, too_big.json()["error"], typo_category.status_code, typo_category.json()["error"]

### Sanity check: does the served model match the offline evaluation?
Here we send 300 cohort resumes through `/v1/match/batch` (in chunks of 64) and compute top-1
accuracy the same way `evaluate_recommendations.ipynb` does. With the real model, the result
should be close to the offline embeddings number (~66% on the full cohort). If it is, the
serving path (cleaning, model, saved job embeddings) hasn't drifted from the offline pipeline.

In [ ]:
from screening_client import ScreeningClient
api = ScreeningClient(http=client, max_batch_size=settings.max_batch_size)

check = resumes_clean.sample(300, random_state=RANDOM_SEED)
results = api.match_batch([{"id": str(i), "text": t} for i, t in zip(check[COL_ID], check.clean_text)], top_k=3)
pred = pd.DataFrame(results)
pred["true"] = check[COL_CATEGORY].values
top1 = (pred.best_category == pred["true"]).mean()
top3 = pd.Series([t in [m["category"] for m in tm] for t, tm in zip(pred["true"], pred.top_matches)]).mean()
print(f"Served model — top-1: {top1:.1%}   top-3: {top3:.1%}   statuses: {pred.status.value_counts().to_dict()}")

## 5. Connecting the ML endpoint to Frontend / Backend squad apps
What each squad gets:

| Squad | Artifact | Notes |
|---|---|---|
| Frontend | `serving/clients/screening_client.js` | Uses `fetch` with no dependencies; typed `ScreeningApiError`; `ERROR_MESSAGES` UI copy |
| Frontend | `serving/demo/index.html` (served at `/demo`) | A working page to copy from |
| Frontend | `/openapi.json` | `npx openapi-typescript http://localhost:8000/openapi.json -o screening.d.ts` |
| Backend | `serving/clients/screening_client.py` | Connection pooling, retries on 503, automatic batch chunking |
| Both | `/docs` | Interactive Swagger UI |

Integration rules: `/v1` paths are stable. Every error has the shape `{"error": {code, message, details, request_id}}`.
Send `X-Request-ID` to trace a call across services. CORS origins come from `SCREENING_CORS_ORIGINS`.
`SCREENING_API_KEY` turns on header auth.

Below, the Backend squad's main call, made through the Python client: rank incoming applicants for a posting.

In [ ]:
applicants = [{"id": f"applicant-{i}", "text": t}
              for i, t in enumerate(resumes_clean.sample(8, random_state=1).clean_text)]
applicants.append({"id": "applicant-blank", "text": ""})

screen = api.screen(
    job={"description": "We are hiring a software engineer to build Python web services, "
                        "databases and cloud infrastructure for our IT department."},
    candidates=applicants,
    top_k=5,
)
print("Rejected:", [(c["id"], c["warnings"][0][:40]) for c in screen["rejected"]])
pd.DataFrame(screen["ranked"])

In [ ]:
# The headers a frontend would read: a request id (echoed back) and the server-side timing.
r = client.get("/v1/categories", headers={"X-Request-ID": "frontend-trace-001", "Origin": "http://localhost:3000"})
{k: r.headers.get(k) for k in ["x-request-id", "x-process-time-ms", "access-control-allow-origin"]}

In [ ]:
# The spec other squads generate types from:
spec = client.get("/openapi.json").json()
print(spec["info"]["title"], spec["info"]["version"])
print("Schemas:", ", ".join(sorted(spec["components"]["schemas"])))

In [ ]:
client.__exit__(None, None, None)   # runs shutdown
print("Done. To serve for real:  cd ../serving && uvicorn app.main:app --port 8000  ->  http://localhost:8000/docs")

---
## Notes
* Latency numbers here come from your machine and include in-process `TestClient`
  overhead. For numbers over real HTTP, run `python ../serving/scripts/benchmark.py` against a live server.
* MiniLM truncates input to ~256 word pieces, the same as Sprint 1. Accuracy parity with
  `evaluate_recommendations.ipynb` depends on that.
* Next steps: chunk long resumes and average the embeddings; tune `SIMILARITY_FLOOR` on
  labelled "no match" examples; use a shared Redis cache once there are several replicas.